# Medidas de Heterogeneidad

In [5]:
import sys
from pathlib import Path
import polars as pl

#Para poder utilizar archivos del proyecto
ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.cleaning.preprocessing import pipeline_preprocesamiento

df = pipeline_preprocesamiento()

In [10]:
columnas = ["nivel_escolaridad", "nom_municipio", "tiene_ahorros_id"]
resultados = []

for col in columnas:
    # Frecuencias relativas (p_i)
    frecuencias = df.group_by(col).len().select(
        p = pl.col("len") / pl.col("len").sum()
    )
    
    k = frecuencias.height
    
    # Expresiones para Gini-Simpson y Entropía de Shannon
    metricas = frecuencias.select([
        (1.0 - (pl.col("p") ** 2).sum()).alias("gini_simpson"),
        (- (pl.col("p") * pl.col("p").log(2)).sum()).alias("shannon_bits")
    ])
    
    gini_simpson = metricas["gini_simpson"][0]
    shannon_bits = metricas["shannon_bits"][0]
    
    # IQV
    iqv = (k / (k - 1)) * gini_simpson if k > 1 else 0.0
    
    resultados.append({
        "Atributo": col,
        "Categorías": k,
        "Gini-Simpson": round(gini_simpson, 6),
        "Variación Cualitativa": round(iqv, 6),
        "Entropía Shannon (bits)": round(shannon_bits, 6)
    })

# Crear el DataFrame de resultados con Polars
tabla_medidas = pl.DataFrame(resultados)

# Configurar vista para mostrar la tabla completa
print(tabla_medidas)

shape: (3, 5)
┌───────────────────┬────────────┬──────────────┬───────────────────────┬─────────────────────────┐
│ Atributo          ┆ Categorías ┆ Gini-Simpson ┆ Variación Cualitativa ┆ Entropía Shannon (bits) │
│ ---               ┆ ---        ┆ ---          ┆ ---                   ┆ ---                     │
│ str               ┆ i64        ┆ f64          ┆ f64                   ┆ f64                     │
╞═══════════════════╪════════════╪══════════════╪═══════════════════════╪═════════════════════════╡
│ nivel_escolaridad ┆ 6          ┆ 0.59883      ┆ 0.718596              ┆ 1.810903                │
│ nom_municipio     ┆ 1206       ┆ 0.995428     ┆ 0.996254              ┆ 8.882818                │
│ tiene_ahorros_id  ┆ 2          ┆ 0.193331     ┆ 0.386661              ┆ 0.495133                │
└───────────────────┴────────────┴──────────────┴───────────────────────┴─────────────────────────┘
